# Module 05 — Explorer le parser silver

**Objectif** : bronze (HTML/summary RSS) → silver (texte lisible) via trafilatura.

**Prérequis**
- Articles en `status=fetched` (`presslake poll`)
- `uv add trafilatura`
- `uv run presslake db init` (migration `silver_s3_uri`)

## Carte module 04 → 05

```
bronze (MinIO)     parse/extract.py (trafilatura)
       │                    │
       └────────────────────┘
                    ▼
            silver (MinIO)
                    ▼
     Postgres status=parsed + silver_s3_uri
```

In [1]:
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

print("Racine:", ROOT)

Racine: /home/anthony-marais/Documents/data_project


## Étape 1 — Lister les articles à parser (`status=fetched`)

In [2]:
from presslake.catalog.articles import list_articles_by_status
from presslake.storage.postgres import get_connection

with get_connection() as conn:
    pending = list_articles_by_status(conn, "fetched", limit=3)

print(f"{len(pending)} article(s) fetched (échantillon)")
for a in pending:
    print(f"  - {a['feed_id']} | {a['title'][:50] if a['title'] else '?'}…")

3 article(s) fetched (échantillon)
  - france24 | Niger : le calme revient à Niamey après la mutiner…
  - france24 | RD Congo : l'opposition rejette les consultations …
  - france24 | Cameroun : Les migrants expulsés des États-Unis dé…


## Étape 2 — Lire un bronze depuis MinIO

In [3]:
from presslake.storage.s3 import get_json_object, get_s3_client, parse_s3_uri

if not pending:
    print("Aucun article fetched — lance: uv run presslake poll")
else:
    art = pending[0]
    bucket, key = parse_s3_uri(art["s3_uri"])
    bronze = get_json_object(get_s3_client(), bucket, key)
    print("feed_id:", bronze["feed_id"])
    print("title:", bronze.get("title"))
    print("summary (début):", (bronze.get("raw") or {}).get("summary", "")[:200])

feed_id: france24
title: Niger : le calme revient à Niamey après la mutinerie
summary (début): 🇳🇪 Au Niger, le calme est de retour à Niamey, après une nuit de vendredi à samedi marquée par une mutinerie déjouée par le régime militaire. Avec l’appui de ses alliés russes, les paramilitaires d’Afr


## Étape 3 — trafilatura sur le summary HTML

In [4]:
from presslake.parse.extract import extract_text_from_bronze

if pending:
    text, source = extract_text_from_bronze(bronze)
    print("source:", source)
    print("texte (début):", text[:400], "…")

source: permalink
texte (début): Pour afficher ce contenu YouTube, il est nécessaire d'autoriser les cookies de mesure d'audience et de publicité.
Accepter
Gérer mes choix
Une extension de votre navigateur semble bloquer le chargement du lecteur vidéo. Pour pouvoir regarder ce contenu, vous devez la désactiver ou la désinstaller.
Niger : le calme revient à Niamey après la mutinerie
Afrique
Publié le : 31/08/2026 - 20:05Modifié le …


## Étape 4 — Construire l'enveloppe silver

In [5]:
import json

from presslake.parse.silver import build_silver_envelope

if pending:
    silver = build_silver_envelope(
        bronze,
        text=text,
        text_source=source,
        bronze_s3_uri=art["s3_uri"],
    )
    print(json.dumps({k: silver[k] for k in ('title', 'canonical_url', 'text_source')}, indent=2, ensure_ascii=False))
    print("len(text):", len(silver["text"]))

{
  "title": "Niger : le calme revient à Niamey après la mutinerie",
  "canonical_url": "https://www.france24.com/fr/vid%C3%A9o/20260831-niger-le-calme-revient-%C3%A0-niamey-apr%C3%A8s-la-mutinerie",
  "text_source": "permalink"
}
len(text): 721


## Étape 5 — Écrire silver dans MinIO (manuel)

In [6]:
from presslake.parse.silver import write_silver_from_bronze
from presslake.storage.s3 import get_bucket

if pending:
    client = get_s3_client()
    silver_uri = write_silver_from_bronze(
        client,
        get_bucket(),
        bronze,
        bronze_s3_uri=art["s3_uri"],
        bronze_key=key,
        text=text,
        text_source=source,
    )
    print("Écrit:", silver_uri)

Écrit: s3://presslake/silver/source=france24/dt=2026-08-31/939006b3a5d63894a4cdd13e72cf16b46157d66d0be4bf461e8febfd2f9e72db.json


## Étape 6 — Commande prod : `presslake parse`

En terminal :
```bash
uv run presslake parse
uv run presslake parse   # 2e fois → 0 parsé
```

In [9]:
from presslake.parse.run import parse_all

# Décommente pour lancer depuis le notebook :
parse_all()


→ 0 article(s) parsé(s)


0

## Étape 7 — Vérifier les statuts Postgres

In [10]:
with get_connection() as conn:
    rows = conn.execute(
        "SELECT status, count(*) FROM articles GROUP BY status ORDER BY status"
    ).fetchall()

for status, n in rows:
    print(f"{status:10} {n}")

parsed     128


## Fichiers prod

| Fichier | Rôle |
|---|---|
| `parse/extract.py` | trafilatura summary / permalink |
| `parse/silver.py` | enveloppe + écriture MinIO |
| `parse/run.py` | boucle fetched → parsed |
| `catalog/articles.py` | `mark_parsed`, `list_articles_by_status` |

Tuto : [`docs/modules/05-parser-silver.md`](../docs/modules/05-parser-silver.md)